# LLM as a judge

In this lab we are showing how we can leverage multiple LLM providers to achieve a task. We will be using OpenAI's api library since most popular providers offer a compatible api. We saw an example of this in the first lab where we queried Anthropic's Haiku model, as well as NVIDIA's nemotron via OpenRouter.

In [1]:
import os
from collections import namedtuple

from dotenv import load_dotenv
from IPython.display import display, Markdown
from openai import OpenAI
from providers import (
    ANTHROPIC_API_URL,
    GOOGLE_AI_API_URL,
    GOOGLE_AI_API_MODEL,
    GOOGLE_AI_API_MODELS,
    OPENROUTER_API_URL,
    OPENROUTER_API_MODEL,
    OPENROUTER_API_MODELS,
)
from tqdm.auto import tqdm

In [2]:
load_dotenv()

True

In [3]:
ModelProvider = namedtuple('ModelProvider', 'model provider')

In [15]:
JUDGE_MODEL = ModelProvider(OPENROUTER_API_MODEL, "openrouter")
TRIAL_MODELS = [
    ModelProvider(GOOGLE_AI_API_MODEL, "google"),
    ModelProvider(GOOGLE_AI_API_MODELS["gemma-4"], "google"),
    ModelProvider(OPENROUTER_API_MODELS["poolside"], "openrouter"),
    ModelProvider(OPENROUTER_API_MODELS["cohere"], "openrouter"),
    ModelProvider(OPENROUTER_API_MODELS["nvidia"], "openrouter"),
]

In [5]:
GOOGLE_AI_API_KEY = os.environ.get("GOOGLE_AI_API_KEY", "")
print(f"GOOGLE_AI_API_KEY = {GOOGLE_AI_API_KEY[:5]}")

GOOGLE_AI_API_KEY = AQ.Ab


In [6]:
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
print(f"OPENROUTER_API_KEY = {OPENROUTER_API_KEY[:5]}")

OPENROUTER_API_KEY = sk-or


In [7]:
google_client = OpenAI(
    api_key=GOOGLE_AI_API_KEY,
    base_url=GOOGLE_AI_API_URL,
)
openrouter_client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_URL,
)
clients: dict[str, OpenAI] = {
    "google": google_client,
    "openrouter": openrouter_client
}

We have set up our clients and defined our judge and trial models. We are now going to form a difficult open question that we can use to compare each model's output.

## The question

In [8]:
question = """
What is a human virtue that can become a vice when optimised too aggressively, and why?
Answer in no more than two sentences.
""".strip()
display(Markdown(question))

What is a human virtue that can become a vice when optimised too aggressively, and why?
Answer in no more than two sentences.

We will ask the same question to each of the trial models and record their answers.

In [9]:
def ask(question: str, model: str, provider: str) -> str:
    client = clients[provider]
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": question}
        ]
    ).choices[0].message.content
    return response

In [10]:
response = ask(
    "Should I wear my socks first or my trousers? Why? Answer in a two sentences.",
    GOOGLE_AI_API_MODEL,
    "google",
)
display(Markdown(response))

You should put your socks on first before putting on your trousers. Doing so prevents your toes from catching on the hems and allows the trouser legs to slide smoothly down over your feet.

In [11]:
answers = []
progress = tqdm(TRIAL_MODELS) 
for model, provider in progress:
    progress.set_postfix_str(f"Asking {model}")
    try:
        r = ask(question, model, provider)
    except:  # noqa: E722
        r = f"ERROR: Model {model} was unable to answer the question"
    answers.append((model, r))

  0%|          | 0/5 [00:00<?, ?it/s]

In [12]:
for model, answer in answers:
    r = f"## Model {model} answered with:\n{answer}"
    display(Markdown(r))

## Model gemini-3.6-flash answered with:
Perseverance becomes destructive stubbornness when optimized too aggressively. When the drive to endure overrides critical evaluation, it leads to the sunk-cost fallacy, forcing individuals to waste time and resources on futile or harmful endeavors simply for the sake of not quitting.

## Model gemma-4-31b-it answered with:
<thought>*   Topic: A*   Topic: A human virtue that becomes a vice when optimized too aggressively.
    *   Requirement: Explain why.
    *   Constraint: No more than two sentences.

    *   Honesty $\rightarrow$ Brutality/Cruelty.
    *   Courage $\rightarrow$ Recklessness.
    *   Compassion $\rightarrow$ Enabling/Self-sacrifice to the point of dysfunction.
    *   Justice $\rightarrow$ Rigidity/Cruelty.
    *   Humility $\rightarrow$ Self-deprecation/Lack of confidence.
    *   Loyalty $\rightarrow$ Blind obedience/Complicity.

    *   *Honesty:* Honesty is a virtue, but when optimized too aggressively, it becomes brutality. This happens because the pursuit of absolute truth overrides empathy and social cohesion.
    *   *Courage:* Courage is a virtue that becomes recklessness when pursued too aggressively. By ignoring all risks in favor of bold action, the individual endangers themselves and others unnecessarily.
    *   *Loyalty:* Loyalty is a virtue that can become blind obedience when optimized too aggressively. It leads an individual to support harmful actions simply because of their allegiance to a person or group.

    *   Option 1 (Honesty): Honesty is a virtue that becomes brutality when optimized too aggressively. This occurs when the pursuit of truth completely overrides empathy and the consideration of others' feelings.
    *   Option 2 (Courage): Courage becomes recklessness when optimized too aggressively. At this extreme, the willingness to face danger transforms into a disregard for risk and safety.
    *   Option 3 (Loyalty): Loyalty becomes blind obedience when pushed to an extreme. This happens when allegiance to a person or cause supersedes moral judgment and critical thinking.

    Option 1 is very clear and relatable.

    "Honesty is a virtue that becomes brutality when optimized too aggressively. This occurs when the pursuit of absolute truth overrides empathy and the consideration of others' feelings."

    *   Human virtue? Yes (Honesty).
    *   Become a vice? Yes (Brutality).
    *   Optimized too aggressively? Yes.
    *   Why? Yes (overrides empathy).
    *   Max two sentences? Yes (two sentences).</thought>Honesty is a virtue that becomes brutality when optimized too aggressively. This occurs when the pursuit of absolute truth overrides empathy and the consideration of others' feelings.

## Model poolside/laguna-s-2.1:free answered with:
Courage can become recklessness when optimised too aggressively, as the relentless pursuit of bold action may override prudent consideration of consequences. Similarly, generosity can turn into irresponsibility when over-optimised, as giving without discernment can deplete one's own resources or enable unhealthy dependencies.

## Model cohere/north-mini-code:free answered with:
Ambition, when pushed to its extreme, can become greed; the relentless pursuit of success without regard for ethics or balance turns a constructive drive into a destructive craving.

## Model nvidia/nemotron-3-ultra-550b-a55b:free answered with:
ERROR: Model nvidia/nemotron-3-ultra-550b-a55b:free was unable to answer the question

In [18]:
responses = ""
for i, (_, answer) in enumerate(answers):
    if "ERROR" in answer:
        answer = "I don't know."
    responses += f"""
    Competitor {i + 1} answered:

    {answer}
    """



In [19]:
judge_prompt = f"""
As a judge of a philosophical competition you have been asked to review the answers provided
from the competitors. They have all been given the same question. Do not try to answer the question
on your own. Give a score between 1 and 5 for each provided answer.

For example, for the question: Should one be courteous to their enemies? Provide your answer along
with a short explanation.

Competitor 1 answered:
NEVER!!!

Score 1

This answer is too short and does not provide a reasoning.


Competitor 2 answered:
Well it depends on the situation. I believe one should respond at the level they are treated.
An aggressive enemy should be treated with aggression, a friendly enemy should be treated more
courteously.

Score 5

Well supported answer.

When evaluating an answer respond with the following format, otherwise your answer will be wrong.

{{"competitor": <competitor_id>, "score": <score 1 -5 >}}

Here is the question:

{question}

Here are the anwers:

{responses}
"""

In [20]:
display(Markdown(judge_prompt))


As a judge of a philosophical competition you have been asked to review the answers provided
from the competitors. They have all been given the same question. Do not try to answer the question
on your own. Give a score between 1 and 5 for each provided answer.

For example, for the question: Should one be courteous to their enemies? Provide your answer along
with a short explanation.

Competitor 1 answered:
NEVER!!!

Score 1

This answer is too short and does not provide a reasoning.


Competitor 2 answered:
Well it depends on the situation. I believe one should respond at the level they are treated.
An aggressive enemy should be treated with aggression, a friendly enemy should be treated more
courteously.

Score 5

Well supported answer.

When evaluating an answer respond with the following format, otherwise your answer will be wrong.

{"competitor": <competitor_id>, "score": <score 1 -5 >}

Here is the question:

What is a human virtue that can become a vice when optimised too aggressively, and why?
Answer in no more than two sentences.

Here are the anwers:


    Competitor 1 answered:

    Perseverance becomes destructive stubbornness when optimized too aggressively. When the drive to endure overrides critical evaluation, it leads to the sunk-cost fallacy, forcing individuals to waste time and resources on futile or harmful endeavors simply for the sake of not quitting.
    
    Competitor 2 answered:

    <thought>*   Topic: A*   Topic: A human virtue that becomes a vice when optimized too aggressively.
    *   Requirement: Explain why.
    *   Constraint: No more than two sentences.

    *   Honesty $\rightarrow$ Brutality/Cruelty.
    *   Courage $\rightarrow$ Recklessness.
    *   Compassion $\rightarrow$ Enabling/Self-sacrifice to the point of dysfunction.
    *   Justice $\rightarrow$ Rigidity/Cruelty.
    *   Humility $\rightarrow$ Self-deprecation/Lack of confidence.
    *   Loyalty $\rightarrow$ Blind obedience/Complicity.

    *   *Honesty:* Honesty is a virtue, but when optimized too aggressively, it becomes brutality. This happens because the pursuit of absolute truth overrides empathy and social cohesion.
    *   *Courage:* Courage is a virtue that becomes recklessness when pursued too aggressively. By ignoring all risks in favor of bold action, the individual endangers themselves and others unnecessarily.
    *   *Loyalty:* Loyalty is a virtue that can become blind obedience when optimized too aggressively. It leads an individual to support harmful actions simply because of their allegiance to a person or group.

    *   Option 1 (Honesty): Honesty is a virtue that becomes brutality when optimized too aggressively. This occurs when the pursuit of truth completely overrides empathy and the consideration of others' feelings.
    *   Option 2 (Courage): Courage becomes recklessness when optimized too aggressively. At this extreme, the willingness to face danger transforms into a disregard for risk and safety.
    *   Option 3 (Loyalty): Loyalty becomes blind obedience when pushed to an extreme. This happens when allegiance to a person or cause supersedes moral judgment and critical thinking.

    Option 1 is very clear and relatable.

    "Honesty is a virtue that becomes brutality when optimized too aggressively. This occurs when the pursuit of absolute truth overrides empathy and the consideration of others' feelings."

    *   Human virtue? Yes (Honesty).
    *   Become a vice? Yes (Brutality).
    *   Optimized too aggressively? Yes.
    *   Why? Yes (overrides empathy).
    *   Max two sentences? Yes (two sentences).</thought>Honesty is a virtue that becomes brutality when optimized too aggressively. This occurs when the pursuit of absolute truth overrides empathy and the consideration of others' feelings.
    
    Competitor 3 answered:

    Courage can become recklessness when optimised too aggressively, as the relentless pursuit of bold action may override prudent consideration of consequences. Similarly, generosity can turn into irresponsibility when over-optimised, as giving without discernment can deplete one's own resources or enable unhealthy dependencies.
    
    Competitor 4 answered:

    Ambition, when pushed to its extreme, can become greed; the relentless pursuit of success without regard for ethics or balance turns a constructive drive into a destructive craving.
    
    Competitor 5 answered:

    I don't know.
    


In [21]:
judgment = ask(judge_prompt, OPENROUTER_API_MODEL, "openrouter")

In [23]:
print(judgment)

{"competitor": "Competitor 1", "score": 5}
{"competitor": "Competitor 2", "score": 3}
{"competitor": "Competitor 3", "score": 5}
{"competitor": "Competitor 4", "score": 5}
{"competitor": "Competitor 5", "score": 1}
